# Sprint Zero, built step by step with the Strands Agents SDK

This notebook rebuilds Sprint Zero's team the way the workshop *Building AI Agents with Strands SDK* teaches it,
one module at a time, using the real code from this repo:

| Step | Workshop module | What you build here |
|---|---|---|
| 1 | Agent loop | One teammate: a Strands `Agent` on Amazon Bedrock with a persona prompt |
| 2 | Tools | Sam, the Scrum Master, with a `@tool` that looks up Scrum terms |
| 3 | Hooks and steering | A `HookProvider` that keeps every teammate in character |
| 4 | Session managers | Memory that survives a restart (`FileSessionManager`; `S3SessionManager` in production) |
| 5 | Multi-agent | The sprint engine picks which teammates react, and each hears the ones before |
| 6 | Evals | Grading a learner's answer against a rubric (Strands Decider 2B, or a Bedrock model) |

**Where to run it:** Amazon SageMaker (JupyterLab), the workshop environment, or any machine with AWS credentials
and Amazon Bedrock model access to Amazon Nova Lite in `us-east-1`. No Bedrock access? Set `USE_BEDROCK = False`
below and scripted stand-in models answer instead, so every cell still runs.

## 0. Setup

In [ ]:
import os, sys, pathlib

# Clone the repo if this notebook was opened on its own.
if not pathlib.Path("sprintzero").exists() and not pathlib.Path("../sprintzero").exists():
    !git clone -q https://github.com/grand1llusion/sprint-zero.git
    os.chdir("sprint-zero")
elif pathlib.Path("../sprintzero").exists():
    os.chdir("..")

%pip install -q -r requirements.txt

In [ ]:
USE_BEDROCK = True          # False: scripted stand-in models, no AWS calls
os.environ.setdefault("AWS_REGION", "us-east-1")
sys.path[:0] = [".", "tests"]

from strands import Agent
from sprintzero.agents import make_model
from test_sprint_zero import ScriptedModel   # the offline stand-in the test suite uses

def model(*scripted_lines, temperature=0.7):
    """Amazon Nova Lite on Bedrock, or a scripted stand-in when USE_BEDROCK is False."""
    return make_model(temperature=temperature) if USE_BEDROCK else ScriptedModel(list(scripted_lines))

## 1. Agent loop: one teammate

In the workshop, an agent is a model plus a system prompt. Here the system prompt is a persona from
`packs/team.yaml`. Marcus, the backend developer, underestimates everything.

In [ ]:
from sprintzero.packs import load_team, load_pack, person
from sprintzero.agents import Cast, clean

team = load_team()
pack = load_pack("flood-alert", team)
cast = Cast(pack, team)
marcus = person(pack, team, "marcus")
prompt = cast._persona_prompt(marcus)
print(prompt)

In [ ]:
marcus_agent = Agent(model=model("Easy, two points. I can have the SMS sign-up done by tomorrow."),
                     system_prompt=prompt, name=marcus["name"], callback_handler=None)
print(clean(marcus_agent("The Product Owner asks: how big is the text-message sign-up story?")))

## 2. Tools: Sam looks up Scrum terms

A `@tool` is a Python function the agent can choose to call. Sam coaches with the glossary's vetted, plain-language
wording instead of the model's memory.

In [ ]:
from sprintzero.hooks import define_term
print(define_term._tool_func("Definition of Done"))

In [ ]:
sam = team["scrum_master"]
sam_agent = Agent(model=model("A Sprint Goal is the one objective for this Sprint; it keeps the team focused."),
                  system_prompt=cast._persona_prompt(person(pack, team, sam["id"])),
                  name=sam["name"], tools=[define_term], callback_handler=None)
reply = sam_agent("In one sentence, explain to a new Product Owner what a Sprint Goal is. Use define_term.")
print(clean(reply))
print("Tools Sam called:", [k for k, v in reply.metrics.tool_metrics.items() if v.call_count])

## 3. Hooks and steering: keep every teammate in character

The workshop's hooks module shows deterministic control around a model. `StayInCharacter` listens for
`AfterModelCallEvent`. If a reply mentions being an AI, mentions scores, uses lists or emojis, or runs long, it sets
`event.retry = True` so Strands throws the reply away and asks again (once per turn). The rules are plain code.

In [ ]:
import inspect
from sprintzero.hooks import StayInCharacter, check_reply
print(inspect.getsource(StayInCharacter))

In [ ]:
# Force a bad first reply with the stand-in model, to watch the hook steer.
hook = StayInCharacter()
steered = Agent(model=ScriptedModel(["As an AI, I'd give that a strong score! 🎉",
                                     "Honestly, a clear Sprint Goal makes my week easier."]),
                system_prompt=prompt, name=marcus["name"], hooks=[hook], callback_handler=None)
print("Reply:", clean(steered("The Product Owner kept the Sprint Goal.")))
print("Hook log:", hook.log)

## 4. Session managers: memory that survives

The web app is stateless, so each teammate's memory lives in a Strands session manager: one per teammate per
learner. Locally that's `FileSessionManager`; in production `S3SessionManager` (set `SPRINTZERO_MEMORY_BUCKET`).
Below, a second `Agent` built from the same session id remembers what the first one heard.

In [ ]:
import tempfile
from strands.session.file_session_manager import FileSessionManager

store = tempfile.mkdtemp()
def priya(*lines):
    return Agent(model=model(*lines), system_prompt="You are Priya Raman, the tech lead. One short sentence.",
                 name="Priya Raman", callback_handler=None,
                 session_manager=FileSessionManager(session_id="demo-priya", storage_dir=store))

priya("Ok.")("Remember this: our Sprint Goal is 'residents get a test warning in English or Spanish'.")
again = priya("Our Sprint Goal is residents getting a test warning in English or Spanish.")   # a fresh object
print("Messages restored from disk:", len(again.messages))
print(clean(again("What is our Sprint Goal?")))

## 5. Multi-agent: the sprint engine routes each event

The workshop's orchestrator delegates to specialists. In Sprint Zero, plain code is the orchestrator: for each
event it picks the one to three people the decision affects (never all nine), and each hears the ones before.
Here is Day 2: the commissioner promises the board a dashboard by Friday.

In [ ]:
from sprintzero.engine import Session
from sprintzero.grading import Grade

inject = pack["injects"][0]
print(inspect.getsource(Session._speakers))
print("Day", inject["day"], "-", inject["title"])
speakers = list(dict.fromkeys([inject["from"]] + inject["react"]))[:3]   # what _speakers returns for this event
print(person(pack, team, inject["from"])["name"], "says:", " ".join(inject["text"].split()))
print("Who reacts:", [person(pack, team, p)["name"] for p in speakers])

In [ ]:
def agent_for(prompt, name):
    return Agent(model=model("Thanks, that keeps the week clear and gives the board something real to see."),
                 system_prompt=prompt, name=name, hooks=[StayInCharacter()], callback_handler=None)

day2 = Cast(pack, team, agent_for)
answer = ("The dashboard goes on the Product Backlog and I'll order it with you next Sprint. "
          "This Sprint we stay on the Sprint Goal: residents get a test warning.")
situation = f"Day {inject['day']}. Commissioner Pruitt said: {inject['text']}"
for line in day2.react(speakers, situation, answer, Grade("strong", 0.9, "demo")):
    print(f"{line['name']} ({line['role']}): {line['text']}")

## 6. Evals: grade the answer against the rubric

Every free-text answer is graded against the scenario's rubric: strong, partial, weak or off topic. With
`STRANDS_DECIDER_URL` set, Strands Decider 2B grades with a calibrated confidence; otherwise a Bedrock model grades.
Grades are labels, never scores; plain code turns them into results.

In [ ]:
from sprintzero.grading import BedrockGrader, DeciderGrader, FallbackGrader, GRADER_PROMPT
from sprintzero.agents import make_agent

offline_grades = iter(["LEVEL: strong\nREASON: Protects the Sprint Goal and moves the request to the backlog.",
                       "LEVEL: weak\nREASON: Adds the dashboard to this Sprint and asks for overtime."])
def grader_agent(system_prompt):
    if USE_BEDROCK:
        return make_agent(system_prompt, model=make_model(temperature=0.0))
    return Agent(model=ScriptedModel([next(offline_grades)]),
                 system_prompt=system_prompt, callback_handler=None)

grader = BedrockGrader(grader_agent)
if os.environ.get("STRANDS_DECIDER_URL"):
    grader = FallbackGrader(DeciderGrader(), grader)

for reply in [answer, "Sure, we'll build the dashboard too, the team can work late."]:
    g = grader.grade(inject["prompt"], inject["rubric"], reply, situation)
    print(f"{g.level:9} via {g.grader}: {reply[:60]}…\n          why: {g.reason}")

## Put it together

Every piece above is what the web app runs for each learner. To play a full Sprint in the terminal:

```bash
python -m sprintzero.play
```

or run the web app with `uvicorn sprintzero.web.server:app --port 8081`. See the README for deploying on Amazon EC2.